# Upgrade to centralized data references diagrams #

This sample will go through all the diagrams in a workflow and upgrade them to use centralized data references.

### Make connections ###

In [1]:
import arcgis
import json
from arcgis.gis.workflowmanager import WorkflowManager

gis = arcgis.gis.GIS(url='https://organizationUrl/portal', username='admin', password='...')

# Search for an existing Workflow Manager Item
item = gis.content.search('title:"Python Sample"')[0]
workflow_manager = WorkflowManager(item)
print("Created connection to workflow manager item")

Created connection to workflow manager item


### Check and Upgrade Diagrams ###

In [3]:
diagrams = workflow_manager.diagrams
upgraded_diagram_ids = []
not_upgraded_diagram_ids = []

for d in diagrams: 
    try: 
        diagram = workflow_manager.diagram(d.diagram_id)
        upgrade_obj = workflow_manager.diagram_upgraded_version(diagram.diagram_id, diagram.diagram_version)

        if upgrade_obj['failedStepIds']:
            print(f'Diagram {d.diagramid} failed to update all the steps in the diagram.')
            not_upgraded_diagram_ids.append(d.diagram_id)
            continue

        if upgrade_obj['failedDataSourceNames']:
            print(f'Diagram {d.diagramid} failed to update all the data sources in the diagram.')
            not_upgraded_diagram_ids.append(d.diagram_id)
            continue
        
        #add 'active'= true to the diagram json in order to set the upgraded version as the active version.
        transformed_diagram = upgrade_obj['transformedDiagram']
        transformed_diagram['active'] = True
        
        updated = workflow_manager.update_diagram(body=transformed_diagram, delete_draft=True)
        upgraded_diagram_ids.append(d.diagram_id)
    except:
        not_upgraded_diagram_ids.append(d.diagram_id)
        print(f'Could not upgrade diagram {d.diagram_id}')

print('Upgraded Diagrams:')
print(upgraded_diagram_ids)


Upgraded Diagrams:
['N-FxYjkrS4ihnRciAY4q1A', '4XkLyfkkSk-vq_2VPARGcQ', '_v3AfXd3RoKsHp1BI1lhfA', 'lU4MEzjlRYmg0Q7nbret5A']
